In [1]:
!unzip /content/labelled_newscatcher_dataset.zip

Archive:  /content/labelled_newscatcher_dataset.zip
  inflating: labelled_newscatcher_dataset.csv  


# Exercises XP: Vector Databases and RAG
Use this guided notebook and fill each TODO before running cells.

## What you'll learn
- Vector search strategies (KNN, ANN) and evaluation.
- Vector database utility (similarity search, RAG).
- Differences between vector DBs, libraries, and plugins.
- Best practices for vector store usage and performance.
- How LMs use context; embedding generation and storage.
- Querying vector stores and applying LMs for QA with retrieved context.

## What you'll build
A functional RAG pipeline with FAISS and ChromaDB, plus QA over retrieved context using a Hugging Face model.

## 0. Setup
Run the install cell once. If your platform needs system deps (e.g., libomp for FAISS), follow instructions in comments.

In [5]:
%pip uninstall -y pydantic pydantic-core pydantic-settings
%pip install "pydantic<2.0" "chromadb==0.3.21" "faiss-cpu>=1.8.0"
%pip install -U "numpy<2" sentence-transformers transformers

Found existing installation: pydantic 2.13.4
Uninstalling pydantic-2.13.4:
  Successfully uninstalled pydantic-2.13.4
Found existing installation: pydantic_core 2.46.4
Uninstalling pydantic_core-2.46.4:
  Successfully uninstalled pydantic_core-2.46.4
Found existing installation: pydantic-settings 2.14.1
Uninstalling pydantic-settings-2.14.1:
  Successfully uninstalled pydantic-settings-2.14.1
  Using cached pydantic-1.10.26-cp312-cp312-manylinux2014_x86_64.manylinux_2_17_x86_64.whl.metadata (155 kB)
INFO: pip is looking at multiple versions of fastapi to determine which version is compatible with other requirements. This could take a while.
INFO: pip is still looking at multiple versions of fastapi to determine which version is compatible with other requirements. This could take a while.
INFO: This is taking longer than usual. You might need to provide the dependency resolver with stricter constraints to reduce runtime. See https://pip.pypa.io/warnings/backtracking for guidance. If you

In [18]:
import os
import json
from pathlib import Path
import numpy as np
import pandas as pd

# faiss : Bibliothèque pour la recherche de similarité efficace et le regroupement de vecteurs denses.
import faiss

# pydantic : Utilisé ici pour la validation de données (nécessaire en version 1.10 pour la compatibilité ChromaDB).
import pydantic
print(f"Pydantic version: {pydantic.__version__}")

# sentence_transformers : Fournit des modèles de pointe pour transformer du texte en vecteurs numériques (embeddings).
from sentence_transformers import SentenceTransformer, InputExample

# chromadb : Base de données vectorielle open-source conçue pour faciliter la construction d'applications d'IA.
import chromadb
from chromadb.config import Settings

# transformers : Bibliothèque incontournable de Hugging Face pour manipuler des modèles de langage (LLMs).
from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline
from IPython.display import display

os.makedirs('cache', exist_ok=True)

Pydantic version: 1.10.26


## 🌟 Exercise 1 · Data loading and preparation

In [19]:
# 1. Chargement des données brutes (CSV de news)
data_path = 'labelled_newscatcher_dataset.csv'
pdf = pd.read_csv(data_path, sep=';')

# 2. Création d'un identifiant unique si absent
# Dans une base de données vectorielle, chaque document doit avoir un ID unique pour être retrouvé.
if id not in pdf.columns:
    pdf['id'] = range(len(pdf))

# 3. Réduction du dataset pour la démonstration
# Travailler sur 1000 lignes permet une exécution rapide tout en restant représentatif.
pdf_subset = pdf.head(1000)

print(f"Dimensions du sous-ensemble : {pdf_subset.shape}")
display(pdf_subset[['id', 'title']].head())
print(pdf_subset.info())

Dimensions du sous-ensemble : (1000, 7)


,id,title
0,0,A closer look at water-splitting's solar fuel ...
1,1,"An irresistible scent makes locusts swarm, stu..."
2,2,Artificial intelligence warning: AI will know ...
3,3,Glaciers Could Have Sculpted Mars Valleys: Study
4,4,Perseid meteor shower 2020: What time and how ...


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1000 entries, 0 to 999
Data columns (total 7 columns):
 #   Column          Non-Null Count  Dtype 
---  ------          --------------  ----- 
 0   topic           1000 non-null   object
 1   link            1000 non-null   object
 2   domain          1000 non-null   object
 3   published_date  1000 non-null   object
 4   title           1000 non-null   object
 5   lang            1000 non-null   object
 6   id              1000 non-null   int64 
dtypes: int64(1), object(6)
memory usage: 54.8+ KB
None


## 🌟 Exercise 2 · Vectorization with Sentence Transformers

In [23]:
from sentence_transformers import InputExample

# 1. Définition d'un exemple d'entrée
# Sentence Transformers utilise l'objet InputExample pour structurer les données.
# 'guid' est l'identifiant, 'texts' contient le contenu textuel à encoder.
def example_create_fn(idx: int, text: str) -> InputExample:
    return InputExample(guid=str(idx), texts=[text], label=0.0)

# 2. Transformation du DataFrame en liste d'objets InputExample
faiss_train_examples = [example_create_fn(idx, text) for idx, text in zip(pdf_subset['id'], pdf_subset['title'])]
print(f'Nombre d\'exemples créés pour l\'encodage : {len(faiss_train_examples)}')

Nombre d'exemples créés pour l'encodage : 1000


In [24]:
# 1. Initialisation du modèle d'Embedding
# 'all-MiniLM-L6-v2' est un modèle léger et performant qui convertit des phrases en vecteurs de 384 dimensions.
model = SentenceTransformer('all-MiniLM-L6-v2')

# 2. Encodage des titres
# Cette étape transforme chaque titre (texte) en une liste de nombres (vecteurs).
titles_list = pdf_subset['title'].tolist()
faiss_title_embedding = model.encode(titles_list, convert_to_numpy=True, show_progress_bar=True)

print(f"Nombre de vecteurs : {len(faiss_title_embedding)}")
print(f"Dimension de chaque vecteur : {len(faiss_title_embedding[0])}")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Batches:   0%|          | 0/32 [00:00<?, ?it/s]

Nombre de vecteurs : 1000
Dimension de chaque vecteur : 384


## 🌟 Exercise 3 · FAISS indexing and search

In [20]:
pdf_to_index = pdf_subset
id_index = pdf_to_index['id'].to_numpy().astype(np.int64)

# 1. Préparation des vecteurs
content_encoded_normalized = faiss_title_embedding.astype('float32')

# 2. Normalisation L2
# Essentiel pour utiliser le produit scalaire (Inner Product) comme mesure de similarité cosinus.
faiss.normalize_L2(content_encoded_normalized)

# 3. Création de l'index FAISS
# IndexFlatIP mesure la similarité par produit scalaire.
# IndexIDMap permet d'associer nos propres IDs (du DataFrame) aux vecteurs.
index_content = faiss.IndexIDMap(faiss.IndexFlatIP(content_encoded_normalized.shape[1]))
index_content.add_with_ids(content_encoded_normalized, id_index)

print(f"Nombre de documents indexés dans FAISS : {index_content.ntotal}")

Nombre de documents indexés dans FAISS : 1000


In [25]:
def search_content(query: str, pdf_to_index: pd.DataFrame, k: int = 3):
    # 1. Encodage de la requête utilisateur
    # On transforme la question en vecteur avec le même modèle que pour les documents.
    query_vector = model.encode([query])
    faiss.normalize_L2(query_vector)

    # 2. Recherche dans l'index FAISS
    # On cherche les 'k' vecteurs les plus proches du vecteur de la requête.
    sims, ids = index_content.search(query_vector.astype('float32'), k)

    # 3. Récupération des données textuelles correspondantes
    results = pdf_to_index[pdf_to_index['id'].isin(ids[0])].copy()
    results['similarities'] = sims[0]
    return results.sort_values(by='similarities', ascending=False)

# Test de la recherche avec un mot-clé
display(search_content('animal', pdf_to_index, k=5))

,topic,link,domain,published_date,title,lang,id,similarities
99,TECHNOLOGY,https://www.gematsu.com/2020/08/ghostwire-toky...,gematsu.com,2020-08-07 16:43:13,Ghostwire: Tokyo confirms dog petting,en,99,0.391902
176,TECHNOLOGY,https://www.pushsquare.com/news/2020/08/random...,pushsquare.com,2020-08-03 16:30:00,Random: You Can Pick Up and Pet Cats in Assass...,en,176,0.376784
762,SCIENCE,https://af.reuters.com/article/worldNews/idAFK...,af.reuters.com,2020-08-13 16:51:00,'Secret' life of sharks: Study reveals their s...,en,762,0.344059
928,SCIENCE,https://www.thecut.com/2020/08/scientists-say-...,thecut.com,2020-08-04 12:52:00,Just Let This Lizard Be a Dinosaur,en,928,0.317387
975,HEALTH,https://www.news-medical.net/news/20200813/Res...,news-medical.net,2020-08-13 05:18:00,Researchers explore social behavior of animals...,en,975,0.295497


## 🌟 Exercise 4 · ChromaDB collection and querying

In [21]:
# 1. Initialisation du client ChromaDB
# ChromaDB gère à la fois le stockage des vecteurs et des métadatas (ex: catégories).
chroma_client = chromadb.Client(Settings(anonymized_telemetry=False))
collection_name = 'my_news'

try:
    chroma_client.delete_collection(name=collection_name)
except:
    pass

# 2. Création de la collection
collection = chroma_client.create_collection(name=collection_name)

# 3. Ajout des données
# On passe les textes (documents), les IDs et les métadonnées pour permettre le filtrage ultérieur.
collection.add(
    documents=pdf_subset['title'].tolist(),
    ids=[str(i) for i in pdf_subset['id'].tolist()],
    metadatas=[{'topic': t} for t in pdf_subset['topic'].tolist()]
)

# 4. Requête de similarité
# ChromaDB s'occupe d'encoder la requête 'space exploration' automatiquement en interne.
results = collection.query(
    query_texts=['space exploration'],
    n_results=3
)
print("Résultats de la recherche dans ChromaDB :")
print(json.dumps(results, indent=2))

ERROR:chromadb.telemetry.posthog:Failed to send telemetry event client_start: capture() takes 1 positional argument but 3 were given


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

ERROR:chromadb.telemetry.posthog:Failed to send telemetry event collection_add: capture() takes 1 positional argument but 3 were given


Résultats de la recherche dans ChromaDB :
{
  "ids": [
    [
      "811",
      "7",
      "654"
    ]
  ],
  "embeddings": null,
  "documents": [
    [
      "The scramble for space at Earth\u2019s outer limits",
      "Orbital space tourism set for rebirth in 2021",
      "Scientific research to widen on ISS as SpaceX eyes commercial crew missions"
    ]
  ],
  "metadatas": [
    [
      {
        "topic": "SCIENCE"
      },
      {
        "topic": "SCIENCE"
      },
      {
        "topic": "SCIENCE"
      }
    ]
  ],
  "distances": [
    [
      0.9223116040229797,
      0.9785246253013611,
      1.089150309562683
    ]
  ]
}


## 🌟 Exercise 5 · Question answering with a Hugging Face model

In [22]:
from transformers import T5ForConditionalGeneration, T5Tokenizer

# 1. Chargement du modèle FLAN-T5
# Un modèle 'text-to-text' capable de répondre à des questions sur la base d'un contexte.
model_id = 'google/flan-t5-small'
tokenizer = T5Tokenizer.from_pretrained(model_id)
model_t5 = T5ForConditionalGeneration.from_pretrained(model_id)

# 2. Préparation du contexte RAG
# On concatène les titres récupérés lors de l'étape précédente pour nourrir le modèle.
question = "What is the latest news on space development?"
context = " ".join(results['documents'][0])

# 3. Construction du Prompt (Instruction + Contexte + Question)
# Le format 'question: ... context: ...' est optimisé pour les modèles T5.
input_text = f"question: {question} context: {context}"
inputs = tokenizer(input_text, return_tensors="pt")

# 4. Génération de la réponse
# Le modèle utilise le contexte fourni pour extraire ou générer la réponse correcte.
outputs = model_t5.generate(**inputs, max_length=128)
response = tokenizer.decode(outputs[0], skip_special_tokens=True)

print(f"Question posée : {question}")
print(f"Réponse du modèle (basée sur le contexte) : {response}")

Loading weights:   0%|          | 0/190 [00:00<?, ?it/s]

[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


Question posée : What is the latest news on space development?
Réponse du modèle (basée sur le contexte) : SpaceX
